In [8]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 6.244998,
	"longitude": -75.57151,
	"start_date": "2026-06-01",
	"end_date": "2026-06-30",
	"hourly": ["wind_direction_10m", "wind_speed_10m"],
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_wind_direction_10m = hourly.Variables(0).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(1).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 6.221441268920898°N -75.55184936523438°E
Elevation: 1473.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  wind_direction_10m  wind_speed_10m
0   2026-06-01 00:00:00+00:00          102.264763        4.236697
1   2026-06-01 01:00:00+00:00           99.090195        4.557236
2   2026-06-01 02:00:00+00:00           94.398621        4.693825
3   2026-06-01 03:00:00+00:00           93.691315        5.591600
4   2026-06-01 04:00:00+00:00           90.000000        5.220000
..                        ...                 ...             ...
715 2026-06-30 19:00:00+00:00           59.470375        8.149847
716 2026-06-30 20:00:00+00:00           72.758453        5.465601
717 2026-06-30 21:00:00+00:00           81.027458        6.924738
718 2026-06-30 22:00:00+00:00           61.189304        8.217153
719 2026-06-30 23:00:00+00:00           67.890503        6.217170

[720 rows x 3 columns]


In [20]:
import matplotlib.pyplot as plt
import numpy as np

theta = hourly_dataframe.iloc[:,1]
v_mag = hourly_dataframe.iloc[:,2]

theta_rad = np.radians(theta)
v1 = - v_mag * np.sin(theta_rad)
v2 = - v_mag * np.cos(theta_rad)

np.savez_compressed(
    "wind_vectors.npz", 
    u=v1.to_numpy(), 
    v=v2.to_numpy()
)